# Use Case — Prioritizing Bus-Stop Cooling Interventions

**Who this is for**  
Urban planners, transit-authority analysts, and climate-adaptation leads who have *their own* infrastructure data (bus stops, public benches, playgrounds, bike-share docks, shelters, schools) and need to decide **which locations to treat first** when budget is limited.

**The scenario**  
Summer heat is making your city's bus stops unbearable. Ridership dips, complaints pile up, and the council has just approved a cooling-intervention budget — trees, shade structures, reflective pavement. You have a list of bus stops. You do **not** have the budget to treat all of them. You need a data-backed, defensible shortlist.

This notebook combines **your data** (a bus-stops point layer) with **FortyGuard's layers** (heatmap, satellite segmentation, street view, environmental parameters) to answer four questions in sequence:

1. **Which stops are actually hot?**  ← heatmap × your points
2. **Why are they hot?**  ← satellite segmentation on the top hotspots
3. **What does that look like on the ground?**  ← street view on the #1 stop
4. **When is heat at its worst here?**  ← environmental parameters profile

The final output is a prioritized action list — one row per stop, ranked by temperature, with a dominant cause and a recommended intervention.

> **Bring your own data.** Put a bus-stops CSV at `data/sample_bus_stops.csv` (the `data/` directory is git-ignored — not shipped). Use the same columns (`stop_id`, `name`, `latitude`, `longitude`) and everything downstream just works; swap the path in Step 1 to use your own.

> **U.S. coverage only.** All FortyGuard endpoints operate over locations inside the United States. Swap the AOI to any U.S. city — coordinates outside the U.S. will return errors or empty responses.

> **Dates: 2021 to today.** `STUDY_DATE` must be on or after `2021-01-01` (the catalog's start) and no later than today; earlier or future dates fail at the heatmap call with a "no data available" error.

---

## Setup

Load `.env`, instantiate the client, define the study area. Run `notebooks/00_setup.ipynb` first if this cell errors out.

In [ ]:
import sys, pathlib, textwrap
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

# Study parameters. Change STUDY_DATE for the day you want to prioritize on.
AOI              = SAN_JOSE_POLYGON      # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE       = '2024-07-15'
STUDY_HOUR       = '14:00'               # design-peak afternoon — required by heatmap & satellite calls
GRANULARITY_M    = 80                   # 100 m → ~10 k tiles over this AOI; 80 m would be ~16 k
TOP_N_TO_DIAGNOSE = 3                    # deepen analysis on this many hottest stops


# --- Quiet polling helper ----------------------------------------------------
# Submits an async API call and waits without printing every status tick.
# You see exactly two lines per call: "Submitted ..." and "✓ ... completed."
#
# The API occasionally returns 403 "Unauthorized access" on the very first
# /v1/status/{id} GET before the activity is fully registered — we tolerate
# that by sleeping briefly and retrying a few times before giving up.
import time as _time
from fortyguard.exceptions import FortyGuardError as _FortyGuardError

def submit_and_wait_quiet(client_method, label, *, poll_interval=8.0,
                          initial_delay=3.0, transient_403_retries=4,
                          failure_retries=2, timeout=600.0,
                          skip_on_failure=False, **kwargs):
    # Some analysis tasks (esp. satellite / street-view) intermittently fail or
    # hang on the backend, and some points have no imagery at all. Retry the whole
    # submit up to failure_retries times on a task-level failure; with
    # skip_on_failure=True, return None once retries/timeout are exhausted so a
    # caller loop can skip that point instead of aborting the whole step.
    def _giveup(exc):
        if skip_on_failure:
            print(f"  ⤼ {label} unavailable — skipping ({type(exc).__name__}).")
            return None
        print(f"  ✗ {label} failed: {exc}")
        raise exc
    for submit_attempt in range(failure_retries + 1):
        activity_id = client_method(wait=False, **kwargs)
        print(f"Submitted {label} → {activity_id}")

        if initial_delay:
            _time.sleep(initial_delay)

        for attempt in range(transient_403_retries + 1):
            try:
                result = client.wait_for(activity_id, poll_interval=poll_interval, timeout=timeout)
                print(f"  ✓ {label} completed.")
                return {"activity_id": activity_id, "result": result}
            except _FortyGuardError as exc:
                msg = str(exc)
                is_403 = "-> 403" in msg or "Unauthorized access" in msg
                is_task_failure = " failed:" in msg and "-> " not in msg
                if is_403 and attempt < transient_403_retries:
                    back_off = 5 * (attempt + 1)
                    print(f"  ⏳ status returned 403 (transient); retrying in {back_off}s…")
                    _time.sleep(back_off)
                    continue
                if is_task_failure and submit_attempt < failure_retries:
                    back_off = 5 * (submit_attempt + 1)
                    print(f"  ↻ task failed (transient backend error); re-submitting in {back_off}s…")
                    _time.sleep(back_off)
                    break  # re-submit via the outer loop
                return _giveup(exc)
            except Exception as exc:
                return _giveup(exc)
    return _giveup(RuntimeError(f"{label}: exhausted all submit attempts"))


# 12-stop spectral ramp (cool blue → hot red) used everywhere temperature is colored.
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)


def temp_color(t, lo, hi):
    """Map a temperature to one of TCM_COLORS by linear interpolation in [lo, hi]."""
    if t is None or lo is None or hi is None or hi == lo:
        return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    idx = min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)
    return TCM_COLORS[idx]


def show_heatmap_summary(features, t_stats, source_label):
    """Visual summary of a heatmap result: stats card + colored histogram + colorbar."""
    temps = [f['properties']['temperature'] for f in features
             if f['properties'].get('temperature') is not None]
    if not temps or t_stats.get('min') is None:
        print('No tile temperatures to summarize.')
        return

    lo, hi, mean = float(t_stats['min']), float(t_stats['max']), float(t_stats['mean'])

    # Wrap long source labels so they fit inside the stats column.
    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    # constrained_layout auto-fits titles, labels and colorbar — no manual spacing collisions.
    fig = plt.figure(figsize=(12, 3.4 + 0.15 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    # --- Stats card (left) -------------------------------------------------
    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    ax0.text(0.0, 0.97 - 0.11 * n_label_lines, f"{len(features):,} tiles",
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')
    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    y = 0.55
    for label, val, color in rows:
        ax0.text(0.0,  y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - 0.07), 0.10, 0.14,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f"{val:.2f} °C", transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= 0.22

    # --- Histogram (middle) ------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    # --- Vertical colorbar (right) ----------------------------------------
    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()


# --- Heat-impact buckets shared by the segmentation breakdown helper. ---
_SEG_HEATING   = {'building', 'buildings', 'road', 'roads', 'pavement', 'rooftop',
                  'rooftops', 'bare', 'earth', 'sidewalk', 'wall', 'car', 'vehicle'}
_SEG_COOLING   = {'tree', 'trees', 'vegetation', 'grass', 'greenery', 'park', 'water'}
_SEG_SKY       = {'sky'}
_CAT_COLORS    = {'heating': '#e03131', 'cooling': '#2b8a3e',
                  'sky': '#f59f00',     '—': '#868e96'}


def _seg_category(cls):
    """Categorize a segmentation class as heating / cooling / sky / other."""
    c = cls.lower().strip()
    if c in _SEG_COOLING: return 'cooling'
    if c in _SEG_SKY:     return 'sky'
    if c in _SEG_HEATING: return 'heating'
    return '—'


def show_segmentation_breakdown(segments, legend, title='Class breakdown'):
    """Visual breakdown of segmentation classes: a stacked composition bar on top,
    sorted horizontal bars below. Bars use legend RGB; right-edge tag shows
    heat impact (heating / cooling / sky)."""
    items   = sorted(segments.items(), key=lambda x: float(x[1]), reverse=True)
    classes = [k for k, _ in items]
    pcts    = [float(v) for _, v in items]
    rgbs    = [tuple(c / 255 for c in legend.get(cls, [128, 128, 128])) for cls in classes]
    cats    = [_seg_category(cls) for cls in classes]

    if not classes:
        print('No segmentation classes to display.')
        return

    heat_total = sum(p for c, p in zip(cats, pcts) if c == 'heating')
    cool_total = sum(p for c, p in zip(cats, pcts) if c == 'cooling')
    sky_total  = sum(p for c, p in zip(cats, pcts) if c == 'sky')

    fig = plt.figure(figsize=(11, 2.4 + len(classes) * 0.55), constrained_layout=True)
    gs  = fig.add_gridspec(2, 1, height_ratios=[1.0, max(2.5, len(classes) * 0.55 + 0.6)])

    ax_top = fig.add_subplot(gs[0])
    cum = 0.0
    for cls, pct, color in zip(classes, pcts, rgbs):
        ax_top.barh(0, pct, left=cum, color=color, edgecolor='white', linewidth=1.5)
        if pct >= 5:
            txt_color = 'white' if sum(color) < 1.5 else '#222'
            ax_top.text(cum + pct / 2, 0, f'{cls} {pct:.0f}%',
                        ha='center', va='center', fontsize=9, fontweight='bold',
                        color=txt_color)
        cum += pct
    ax_top.set_xlim(0, 100); ax_top.set_ylim(-0.6, 0.6); ax_top.set_yticks([])
    ax_top.set_title(
        f"Scene composition  —  "
        f"heating {heat_total:.0f}% · cooling {cool_total:.0f}% · sky {sky_total:.0f}%",
        fontsize=11, pad=8)
    for sp in ('top', 'right', 'left'):
        ax_top.spines[sp].set_visible(False)

    ax = fig.add_subplot(gs[1])
    y = list(range(len(classes)))
    bars = ax.barh(y, pcts, color=rgbs, edgecolor='#333', linewidth=0.6)
    xmax = max(pcts)
    cat_transform = ax.get_yaxis_transform()

    for i, (pct, cat) in enumerate(zip(pcts, cats)):
        ax.text(pct + xmax * 0.015, i, f'{pct:.2f}%',
                va='center', ha='left',
                fontsize=10, fontweight='bold', color='#222')
        ax.text(0.99, i, cat,
                va='center', ha='right',
                fontsize=9, color=_CAT_COLORS[cat], fontweight='bold',
                transform=cat_transform)

    ax.set_yticks(y); ax.set_yticklabels(classes, fontsize=10)
    ax.set_xlabel('Coverage (%)')
    ax.set_xlim(0, xmax * 1.35)
    ax.set_title(title, pad=10)
    ax.invert_yaxis()
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    ax.grid(axis='x', alpha=0.25, linestyle='--')

    plt.show()


print(f'Authenticated to {client.base_url}')
print(f'Study window: {STUDY_DATE} (single-day analysis, full 24 h)')

---
## Step 1 — Load your data

### What you are doing
Reading a bus-stops point layer from CSV. The schema is minimal — `stop_id`, `name`, `latitude`, `longitude` — so you can export this directly from the transit agency's GIS, a GTFS feed, or a spreadsheet.

### Why this matters
Everything downstream is built around the geometry of **your** assets. By starting from your own data, the outputs land in your existing workflow: same IDs, same names, same coordinate system. That is the difference between a dashboard and something the operations team can act on.

In [ ]:
# Swap this path for your own CSV — same four columns.
stops = pd.read_csv(ROOT / 'data' / 'sample_bus_stops.csv')
print(f'Loaded {len(stops)} stops')
stops.head()

In [ ]:
# Plot the stops as a quick sanity check.
center = [stops['latitude'].mean(), stops['longitude'].mean()]
fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
for _, r in stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=5, color='#1f77b4', fill=True, fill_opacity=0.9,
        popup=f"{r.stop_id} — {r['name']}",
    ).add_to(fmap)
folium.GeoJson(AOI, style_function=lambda _: {'color': '#555', 'fill': False, 'weight': 1, 'dashArray': '5,5'}).add_to(fmap)
fmap.fit_bounds([[stops['latitude'].min(), stops['longitude'].min()],
                 [stops['latitude'].max(), stops['longitude'].max()]])
fmap

---
## Step 2a — Create the heat layer (via API)

### What you are doing
Requesting a high-resolution heatmap over the study AOI at the design-peak hour. The response is a GeoJSON tile layer with a temperature value on every tile.

### Why this matters
Weather stations give you one number for the whole city. A heatmap gives you temperature **at the spatial resolution your decisions are made** — block by block. That is what lets you separate hot stops from merely-average stops.

> Run **either** Step 2a (live API call, consumes credits) **or** Step 2b (load a cached sample, free). Both produce the same `map_data` / `features` / `t_stats` variables, so everything downstream works identically.

In [ ]:
import json

heatmap = submit_and_wait_quiet(
    client.create_heatmap,
    f"heatmap {STUDY_DATE}",
    polygon_aoi=AOI,
    start_date=STUDY_DATE,
    start_time=STUDY_HOUR,
    filter_type=3,                   # single day — daily aggregate per tile
    granularity=GRANULARITY_M,
)

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

# --- Persist the raw response under data/heatmaps/ for reuse / cache. ---
HEATMAP_DIR = ROOT / 'data' / 'heatmaps'
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)
LIVE_HEATMAP_PATH = HEATMAP_DIR / f"heatmap_san_jose_{STUDY_DATE}_live.geojson"
with open(LIVE_HEATMAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(map_data, f)
print(f"Saved raw heatmap → {LIVE_HEATMAP_PATH.relative_to(ROOT)}")

# Single-day responses carry per-tile aggregates (min/max/average_temperature) in °C.
# Use the daily MAX (peak) — that's what every downstream signal should reflect:
# the design-peak temperature each stop is exposed to. Daily average drags in the
# cool nighttime hours and would underweight afternoon heat exposure.
def _as_c(t):
    return None if t is None else round(float(t), 2)

for feat in features:
    props = feat.setdefault('properties', {})
    props['temperature'] = _as_c(props.get('max_temperature'))

# Build t_stats from the converted peak temperatures.
temps = [f['properties']['temperature'] for f in features
         if f['properties'].get('temperature') is not None]
t_stats = {
    'count': len(temps),
    'min':   round(min(temps), 2) if temps else None,
    'max':   round(max(temps), 2) if temps else None,
    'mean':  round(sum(temps) / len(temps), 2) if temps else None,
}

print(f"Daily peak per tile · {len(features)} tiles · "
      f"{t_stats['min']}–{t_stats['max']} °C")
show_heatmap_summary(features, t_stats, f"Live API · {STUDY_DATE} (daily peak)")

---
## Step 2b — Or: load a pre-generated heatmap (for testing)

### What you are doing
Loading a cached heatmap from `data/san_jose_heatmap_sample.geojson` instead of calling the API. Tile properties (`min_temperature`, `max_temperature`, `average_temperature`, in °C) are reduced to a single `temperature` property (the daily peak) so the rest of the notebook sees the same shape Step 2a would produce.

### Why this matters
Use this path when iterating on the downstream logic without burning API credits on a heatmap you already have. Skip it on a real run — Step 2a gives you a heatmap for the exact date, hour, and AOI you care about.

In [ ]:
import json

# Cached heatmaps live under data/heatmaps/. Step 2a writes new live captures
# here too — change the filename below to load any captured run.
HEATMAP_DIR  = ROOT / 'data' / 'heatmaps'
HEATMAP_PATH = HEATMAP_DIR / 'san_jose_heatmap_sample.geojson'

with open(HEATMAP_PATH, 'r') as f:
    map_data = json.load(f)

# Normalize each feature so it carries a single `temperature` (°C) property,
# mirroring what client.create_heatmap(...) would return. Use the daily MAX
# (peak) — same choice as Step 2a, so downstream rank/diagnose see the same
# temperatures whether we ran live or loaded the cache.
def _as_c(t):
    return None if t is None else round(float(t), 2)

features = map_data.get('features', [])
for feat in features:
    props = feat.setdefault('properties', {})
    props['temperature'] = _as_c(props.get('max_temperature'))

temps = [f['properties']['temperature'] for f in features if f['properties'].get('temperature') is not None]
t_stats = {
    'count': len(temps),
    'min': round(min(temps), 2) if temps else None,
    'max': round(max(temps), 2) if temps else None,
    'mean': round(sum(temps) / len(temps), 2) if temps else None,
}

show_heatmap_summary(features, t_stats, f"Cached · {HEATMAP_PATH.name} (daily peak)")

---
## Step 3 — Correlate your data with the heat layer

### What you are doing
For each bus stop, finding the heatmap tile that contains it and copying that tile's temperature onto the stop. This is the **spatial join** — the moment where your asset layer and our thermal layer merge into one table.

### Why this matters
Before this step, "it's hot in the city" was a general observation. After this step, every row in your bus-stops table carries a specific temperature at the design hour. You can now sort, filter, group by route, report by neighborhood — anything you would normally do with your operational data.

In [ ]:
# Build shapely geometries once, then assign each stop its containing tile temperature.
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _stop_temperature(lat: float, lon: float):
    p = Point(lon, lat)
    # Preferred: tile that contains the point.
    for poly, temp in tile_polys:
        if poly.contains(p):
            return temp
    # Fallback: nearest tile by centroid distance.
    if not tile_polys:
        return None
    nearest = min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))
    return nearest[1]

stops['temperature_c'] = stops.apply(
    lambda r: _stop_temperature(r['latitude'], r['longitude']), axis=1
)
stops[['stop_id', 'name', 'temperature_c']].head()

---
## Step 4 — Rank and visualize

### What you are doing
Sorting stops by the temperature value we just attached, then plotting them on a map with the heatmap tiles as the backdrop. Marker color scales with stop temperature; hottest stops jump out visually and land at the top of the table.

### Why this matters
This is the first concrete deliverable — a ranked short-list of candidate locations. Even without the downstream diagnostic steps, this alone is more actionable than any citywide average the council has seen.

In [ ]:
ranked = stops.sort_values('temperature_c', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)
ranked

In [ ]:
# Map: heatmap tiles in the background, bus stops colored by temperature on top.
# Both tiles and stops use the shared spectral ramp (TCM_COLORS) for consistency.
lo, hi = ranked['temperature_c'].min(), ranked['temperature_c'].max()
aoi_mean = t_stats.get('mean')

# --- Enrich each tile's properties so popups/tooltips have useful fields ---
# (idempotent: re-running this cell overwrites the same keys cleanly)
sorted_idxs = sorted(
    (i for i, f in enumerate(features) if f['properties'].get('temperature') is not None),
    key=lambda i: -features[i]['properties']['temperature'],
)
n_ranked = len(sorted_idxs)
for rank, idx in enumerate(sorted_idxs, start=1):
    f = features[idx]; p = f['properties']; t = p['temperature']
    c = shape(f['geometry']).centroid
    p['tile_id']         = f"T{idx:05d}"
    p['tile_rank']       = f"{rank} / {n_ranked}"
    p['temperature_str'] = f"{t:.2f} °C"
    p['delta_from_mean'] = (f"{t - aoi_mean:+.2f} °C"
                            if aoi_mean is not None else "—")
    p['centroid']        = f"{c.y:.5f}, {c.x:.5f}"


def _tile_style(feat):
    t = feat['properties'].get('temperature', lo)
    return {'fillColor': temp_color(t, lo, hi),
            'color': '#00000000', 'fillOpacity': 0.55, 'weight': 0}

def _tile_highlight(_feat):
    # Border + bump opacity so the hovered tile stands out before clicking.
    return {'color': '#222', 'weight': 1.6, 'fillOpacity': 0.85}

def _stop_color(t):
    return '#888' if t is None else temp_color(t, lo, hi)

POPUP_FIELDS  = ['tile_id', 'tile_rank', 'temperature_str', 'delta_from_mean', 'centroid']
POPUP_ALIASES = ['Tile', 'Rank (hottest)', 'Temperature', 'Δ vs AOI mean', 'Centroid (lat, lon)']

fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if features:
    folium.GeoJson(
        map_data,
        style_function=_tile_style,
        highlight_function=_tile_highlight,
        tooltip=folium.GeoJsonTooltip(
            fields=['temperature_str', 'tile_rank'],
            aliases=['Temperature', 'Rank'],
            sticky=True,
        ),
        popup=folium.GeoJsonPopup(
            fields=POPUP_FIELDS, aliases=POPUP_ALIASES, labels=True,
        ),
    ).add_to(fmap)

for _, r in ranked.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap)
fmap.fit_bounds([[ranked['latitude'].min(), ranked['longitude'].min()],
                 [ranked['latitude'].max(), ranked['longitude'].max()]])
fmap

---
## Step 5 — Zoom in on above-average hotspots

### What you are doing
Filtering both the heatmap tiles and the bus stops to the band `mean < temperature ≤ max` and redrawing the same map. The cooler half of the AOI falls away; only the genuinely-hot tiles and the stops inside them remain.

### Why this matters
When every tile is drawn, the eye gets pulled to whatever is warmest in the visible frame — which may still be near the citywide average. Filtering to above-mean sharpens the question: *of the stops that are hotter than typical for the AOI at this hour, where are they clustered?* That's the cluster map the council should see first.

In [ ]:
mean_t = t_stats['mean']
max_t  = t_stats['max']

hot_features = [
    f for f in features
    if f['properties'].get('temperature') is not None
    and f['properties']['temperature'] > mean_t
    and f['properties']['temperature'] <= max_t
]

hot_stops = ranked[(ranked['temperature_c'] > mean_t) &
                   (ranked['temperature_c'] <= max_t)].copy()

# Keep only the tiles that intersect at least one hot stop.
# Use a tiny buffer around each point so boundary points are not missed.
stop_points = [Point(r.longitude, r.latitude).buffer(1e-6) for _, r in hot_stops.iterrows()]
hot_features = [
    f for f in hot_features
    if any(shape(f['geometry']).intersects(p) for p in stop_points)
]
hot_map_data = {'type': 'FeatureCollection', 'features': hot_features}

print(f'AOI mean: {mean_t:.2f} °C   max: {max_t:.2f} °C')
print(f'Tiles intersecting hot stops: {len(hot_features)}')
print(f'Stops above mean: {len(hot_stops)} / {len(ranked)}')

fmap_hot = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if hot_features:
    # Reuses _tile_style / _tile_highlight / POPUP_FIELDS / POPUP_ALIASES from Step 4.
    folium.GeoJson(
        hot_map_data,
        style_function=_tile_style,
        highlight_function=_tile_highlight,
        tooltip=folium.GeoJsonTooltip(
            fields=['temperature_str', 'tile_rank'],
            aliases=['Temperature', 'Rank'],
            sticky=True,
        ),
        popup=folium.GeoJsonPopup(
            fields=POPUP_FIELDS, aliases=POPUP_ALIASES, labels=True,
        ),
    ).add_to(fmap_hot)
for _, r in hot_stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap_hot)
if len(hot_stops):
    fmap_hot.fit_bounds([[hot_stops['latitude'].min(), hot_stops['longitude'].min()],
                         [hot_stops['latitude'].max(), hot_stops['longitude'].max()]])
fmap_hot

---
## Step 6 — Diagnose the top hotspots (why are they hot?)

### What you are doing
Running satellite segmentation on the top-N hottest stops. The API classifies the surroundings of each point into surface classes (rooftops, roads, vegetation, water, bare land). We collect the percentages into the same DataFrame.

### Why this matters
Knowing a stop is hot is not actionable on its own — *intervention selection depends on the cause*. Planting trees fixes a low-vegetation problem; reflective paving fixes a high-impervious problem; a shade structure fixes a high sky-exposure problem. Satellite segmentation tells you which of those is the dominant driver for each candidate stop.

In [ ]:
import base64, io, json
from PIL import Image

IMPERVIOUS_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGETATION_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segments: dict, keys: set) -> float:
    total = 0.0
    for cls, pct in segments.items():
        if any(k in cls.lower() for k in keys):
            try:
                total += float(pct)
            except (TypeError, ValueError):
                pass
    return round(total, 1)

def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

# Persist raw responses under data/satellite/ — one file per stop, reusable as cache.
SAT_SEG_DIR = ROOT / 'data' / 'satellite'
SAT_SEG_DIR.mkdir(parents=True, exist_ok=True)

top = ranked.head(TOP_N_TO_DIAGNOSE).copy()
impervious, vegetation, raw_segments = [], [], []
sat_results = []  # full responses, kept for the map below

for _, r in top.iterrows():
    sat = submit_and_wait_quiet(
        client.satellite_segmentation,
        f"satellite #{r['rank']} {r['stop_id']}",
        latitude=r.latitude, longitude=r.longitude,
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3,                # single day — full 24 h (start_time ignored)
        granularity=GRANULARITY_M,
    )
    res     = sat['result']

    # Persist the raw response.
    out_path = SAT_SEG_DIR / f"satellite_{r['stop_id']}_{STUDY_DATE}_live.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f"  saved → {out_path.relative_to(ROOT)}")

    seg     = res.get('segmentation', {}) or {}
    segs    = seg.get('segments', {}) or {}
    legend  = seg.get('image_legend', {}) or {}
    raw_segments.append(segs)
    sat_results.append(res)
    impervious.append(_bucket(segs, IMPERVIOUS_KEYS))
    vegetation.append(_bucket(segs, VEGETATION_KEYS))

    # Per-stop visualization — original + segmented image side by side, then class breakdown.
    # `orignal_image` (sic) is the API field; `original_image` is the corrected alias.
    original_img  = _decode_b64(res.get('orignal_image') or res.get('original_image'))
    segmented_img = _decode_b64(seg.get('image_content'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if original_img is not None:  axes[0].imshow(original_img)
    axes[0].set_title(f"#{r['rank']} {r['stop_id']} satellite ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if segmented_img is not None: axes[1].imshow(segmented_img)
    axes[1].set_title("Segmented image")
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

    show_segmentation_breakdown(
        segs, legend,
        title=f"Satellite class breakdown — #{r['rank']} {r['stop_id']}",
    )

top['impervious_pct'] = impervious
top['vegetation_pct'] = vegetation

# --- Map of all diagnosed stops with their tile footprint. ---
fmap_sat = folium.Map(
    location=[top['latitude'].mean(), top['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{r['rank']} {r['stop_id']} — {r['name']}<br>"
               f"{r.temperature_c:.1f} °C<br>"
               f"impervious {r.impervious_pct}% · vegetation {r.vegetation_pct}%"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_sat)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f"~{GRANULARITY_M}m tile",
    ).add_to(fmap_sat)

from IPython.display import display
display(top[['rank', 'stop_id', 'name', 'temperature_c', 'impervious_pct', 'vegetation_pct']])
fmap_sat

---
## Step 6b — Load cached satellite segmentation for a hot tile

### What you are doing
Reading a pre-saved satellite segmentation result from `data/satellite_segmentation_urban_planner.json`. This file was produced by running satellite segmentation at the centroid of one of the hot tiles identified in Step 5. We display the coordinates, the original satellite image, the segmented image, the segmentation percentages, and the location on a map.

### Why this matters
Use this path when you already have a cached segmentation result and want to inspect it without burning API credits. The file contains the original image, the segmented image, and the class breakdown — everything you need to understand why a tile is hot.

In [ ]:
import json, base64, io
from PIL import Image

# Cached satellite segmentation results live under data/satellite/.
# Step 6 writes new live captures here too — change the filename below to load any captured run.
SAT_SEG_DIR  = ROOT / 'data' / 'satellite'
SAT_SEG_PATH = SAT_SEG_DIR / 'satellite_segmentation_urban_planner.json'

with open(SAT_SEG_PATH, 'r') as f:
    sat_data = json.load(f)

# --- Coordinates ---
coords = sat_data['coordinates']
lat, lon = float(coords['latitude']), float(coords['longitude'])
print(f"Location  : ({lat}, {lon})")
print(f"Image year: {sat_data.get('image_year', 'N/A')}")

# --- Segmentation results ---
seg      = sat_data['segmentation']
segments = seg['segments']
legend   = seg.get('image_legend', {})

# --- Decode and show original + segmented images side by side ---
def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

original_img  = _decode_b64(sat_data.get('orignal_image'))
segmented_img = _decode_b64(seg.get('image_content'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if original_img is not None:  axes[0].imshow(original_img)
axes[0].set_title(f"Original satellite image ({lat:.4f}, {lon:.4f})")
axes[0].axis('off')
if segmented_img is not None: axes[1].imshow(segmented_img)
axes[1].set_title("Segmented image")
axes[1].axis('off')
plt.tight_layout(); plt.show()

# --- Visual class breakdown (replaces the text dumps + RGB legend) ---
show_segmentation_breakdown(segments, legend, title='Satellite class breakdown')

# --- Show location on map ---
fmap_seg = folium.Map(location=[lat, lon], zoom_start=16, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Satellite segmentation<br>({lat:.6f}, {lon:.6f})",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_seg)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_seg)

# --- Populate top / impervious_pct / vegetation_pct for Step 9 (cached path). ---
# The cached segmentation covers one tile, so we apply its percentages to the
# top-N hottest stops as a demonstration — on the live path Step 6 runs
# segmentation per-stop and each row gets its own numbers.
IMPERVIOUS_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGETATION_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segs, keys):
    total = 0.0
    for cls, pct in segs.items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

top = ranked.head(TOP_N_TO_DIAGNOSE).copy()
top['impervious_pct'] = _bucket(segments, IMPERVIOUS_KEYS)
top['vegetation_pct'] = _bucket(segments, VEGETATION_KEYS)
print(f"\nPopulated top (n={len(top)}) for Step 9 — "
      f"impervious {top['impervious_pct'].iloc[0]}%, vegetation {top['vegetation_pct'].iloc[0]}%")

fmap_seg

---
## Step 7 — Ground-truth the #1 stop with street view

### What you are doing
Running street view segmentation at the hottest stop, oriented down the street. The API returns a ground-level image plus a pixel-wise segmentation.

### Why this matters
Satellite view shows *surroundings from above*. Street view shows *what a rider waiting at that stop actually sees*. That perspective is where you confirm whether a shade structure is feasible, whether there is room for trees, and whether the shelter itself (a metal box that absorbs heat) is part of the problem.

In [ ]:
import base64, io, json
from PIL import Image

def _decode(b64):
    if not b64: return None
    if isinstance(b64, list): b64 = b64[0]
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

# Persist raw responses under data/street_view/ — one file per stop, reusable as cache.
STREET_SEG_DIR = ROOT / 'data' / 'street_view'
STREET_SEG_DIR.mkdir(parents=True, exist_ok=True)

sky_pct_list, building_pct_list = [], []
sv_results = []  # kept for the per-stop map below

for _, r in top.iterrows():
    sv = submit_and_wait_quiet(
        client.street_view_segmentation,
        f"streetview #{r['rank']} {r['stop_id']}",
        latitude=r.latitude, longitude=r.longitude,
        vertical_angle=5.0, horizontal_angle=0.0, back_view=False,
        skip_on_failure=True, timeout=240,
    )
    if sv is None:   # no street-view imagery / task failed — record blanks and skip
        sv_results.append({})
        sky_pct_list.append(None)
        building_pct_list.append(None)
        continue
    res = sv['result']

    # Persist the raw response.
    out_path = STREET_SEG_DIR / f"streetview_{r['stop_id']}_{STUDY_DATE}_live.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f"  saved → {out_path.relative_to(ROOT)}")

    front          = res.get('front', {}) or {}
    front_segments = front.get('segments', {}) or {}
    front_legend   = front.get('image_legend', {}) or {}
    sv_results.append(front)

    # Decode + show original/segmented side by side.
    original_img  = _decode(front.get('original_image'))
    segmented_img = _decode(front.get('segmented_image'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if original_img is not None:  axes[0].imshow(original_img)
    axes[0].set_title(f"#{r['rank']} {r['stop_id']} street view ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if segmented_img is not None: axes[1].imshow(segmented_img)
    axes[1].set_title("Segmentation")
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

    show_segmentation_breakdown(
        front_segments, front_legend,
        title=f"Street view class breakdown — #{r['rank']} {r['stop_id']}",
    )

    sky_pct_list.append(_bucket(front_segments, {'sky'}))
    building_pct_list.append(_bucket(front_segments, {'building', 'buildings', 'wall'}))

top['sky_pct']      = sky_pct_list
top['building_pct'] = building_pct_list

# Step 9 still consumes hot1's sky_pct as a scalar — keep that name for compat.
sky_pct      = top['sky_pct'].iloc[0]
building_pct = top['building_pct'].iloc[0]

print(f"\nPer-stop street-view drivers:")
for _, r in top.iterrows():
    print(f"  #{r['rank']} {r['stop_id']}: sky {r.sky_pct}% · building {r.building_pct}%")

# --- Map of all diagnosed stops. ---
fmap_street = folium.Map(
    location=[top['latitude'].mean(), top['longitude'].mean()],
    zoom_start=15, tiles='cartodbpositron',
)
for _, r in top.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{r['rank']} {r['stop_id']} — {r['name']}<br>"
               f"sky {r.sky_pct}% · building {r.building_pct}%"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_street)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f"~{GRANULARITY_M}m tile",
    ).add_to(fmap_street)

# Keep `hot1` defined so Step 8 still finds it.
hot1 = top.iloc[0]

fmap_street

---
## Step 7b — Load cached street view segmentation for a hot tile

### What you are doing
Reading a pre-saved street view segmentation result from `data/street_view_segmentation_urban_planner.json`. This file was produced by running street view segmentation at the centroid of one of the hot tiles identified earlier. We display the coordinates, the original street view image, the segmented image, the segmentation percentages, and the location on a map.

### Why this matters
Use this path when you already have a cached street view result and want to inspect it without burning API credits. The file contains the ground-level original image, the pixel-wise segmented image, and the class breakdown — everything you need to confirm on-the-ground conditions at a hotspot.

In [ ]:
import json, base64, io
from PIL import Image

# Cached street-view segmentation results live under data/street_view/.
# Step 7 writes new live captures here too — change the filename below to load any captured run.
STREET_SEG_DIR  = ROOT / 'data' / 'street_view'
STREET_SEG_PATH = STREET_SEG_DIR / 'street_view_segmentation_urban_planner.json'

with open(STREET_SEG_PATH, 'r') as f:
    street_data = json.load(f)

# --- Coordinates ---
coords = street_data['coordinates']
lat, lon = float(coords['latitude']), float(coords['longitude'])
front = street_data['front']
print(f"Location  : ({lat}, {lon})")
print(f"Image date: {front.get('image_date', 'N/A')}")

# --- Segmentation results ---
segments = front['segments']
legend   = front.get('image_legend', {})

# --- Decode and show original + segmented images side by side ---
def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str)))

original_img  = _decode_b64(front.get('original_image'))
segmented_img = _decode_b64(front.get('segmented_image'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if original_img is not None:  axes[0].imshow(original_img)
axes[0].set_title(f"Original street view ({lat:.4f}, {lon:.4f})")
axes[0].axis('off')
if segmented_img is not None: axes[1].imshow(segmented_img)
axes[1].set_title("Segmented image")
axes[1].axis('off')
plt.tight_layout(); plt.show()

# --- Visual class breakdown (replaces the text dumps + RGB legend) ---
show_segmentation_breakdown(segments, legend, title='Street view class breakdown')

# --- Show location on map ---
fmap_street = folium.Map(location=[lat, lon], zoom_start=17, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Street view segmentation<br>({lat:.6f}, {lon:.6f})",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_street)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_street)

# --- Populate sky_pct for Step 9 (cached-path compatibility). ---
def _bucket(segs, keys):
    total = 0.0
    for cls, pct in segs.items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

sky_pct = _bucket(segments, {'sky'})
print(f"\nsky_pct (for Step 9): {sky_pct}%")

fmap_street

---
## Step 8 — Environmental drivers through the day

### What you are doing
Profiling environmental parameters at the #1 stop from 07:00 to 19:00 on the study date. We plot heat index and relative humidity through the day so you can see *when* discomfort peaks, not just how hot it gets at one instant.

### Why this matters
A stop that is unbearable from 12:00–17:00 demands different intervention timing than one that peaks during evening commute. The hour-by-hour profile tells you whether shade, misting, or ventilation is the right fix — and whether the fix needs to be passive (works all day) or active (runs only during peak).

In [ ]:
import json

# filter_type=3 returns the full 24-hour diurnal series for env_params, so the
# peak-hour detection runs across all 24 samples. We loop the call over the top-N
# diagnosed stops so every row in `top` carries its own peak heat-index, RH, and
# wet-bulb — same per-stop treatment as satellite (Step 6) and street view (Step 7).

# Persist raw responses under data/env_params/ — one file per stop, reusable as cache.
ENV_DIR = ROOT / 'data' / 'env_params'
ENV_DIR.mkdir(parents=True, exist_ok=True)


def _gauge(ax, value, zones, value_label, title, xlim, unit_suffix):
    """Horizontal gauge: colored zones on bottom 60%, marker + value on top 40%."""
    for lo_z, hi_z, color, label in zones:
        ax.axvspan(lo_z, hi_z, color=color, alpha=0.75, ymin=0, ymax=0.6)
        ax.text((lo_z + hi_z) / 2, 0.3, label, ha='center', va='center', fontsize=8)
    if value is not None and value >= 0:
        ax.plot([value], [0.78], marker='v', markersize=16, color='black', zorder=10)
        ax.text(value, 0.93, f'{value}{unit_suffix}', ha='center', fontsize=10, fontweight='bold')
    ax.set_xlim(*xlim); ax.set_ylim(0, 1); ax.set_yticks([])
    ax.set_xlabel(value_label); ax.set_title(title)


def _show_env_for_stop(env_df, solar_cs, tile_temp, label):
    """Per-stop visual: diurnal line plot + rich peak-hour snapshot."""
    plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                             'wet_bulb_temperature_celsius', 'relative_humidity_percent')
                 if c in env_df.columns]
    if plot_cols and len(env_df) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        thermal_cols = [c for c in plot_cols if 'humidity' not in c]
        if thermal_cols:
            env_df[thermal_cols].plot(ax=axes[0], marker='o')
            axes[0].set_title(f"Thermal comfort — {label}")
            axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
        if 'relative_humidity_percent' in env_df.columns:
            env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
            axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

    if 'heat_index_celsius' not in env_df.columns or not len(env_df):
        return None, None, None, None

    peak_idx = env_df['heat_index_celsius'].idxmax()
    row      = env_df.loc[peak_idx]
    peak_hi  = float(row.get('heat_index_celsius'))
    peak_rh  = (float(row.get('relative_humidity_percent'))
                if 'relative_humidity_percent' in env_df.columns else None)
    peak_wb  = (float(row.get('wet_bulb_temperature_celsius'))
                if 'wet_bulb_temperature_celsius' in env_df.columns else None)

    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1.1, 1.1], hspace=0.55, wspace=0.28)

    _gauge(fig.add_subplot(gs[0, 0]),
           value=row.get('heat_index_celsius'),
           zones=[(15, 27, '#b7e4c7', 'Safe'),
                  (27, 32, '#ffd43b', 'Caution'),
                  (32, 41, '#fd7e14', 'Extreme\ncaution'),
                  (41, 54, '#fa5252', 'Danger')],
           value_label='Heat index (°C) — NWS heat-stress scale',
           title=f"Peak hour: {peak_idx.strftime('%Y-%m-%d %H:%M')} — {label}",
           xlim=(15, 54), unit_suffix=' °C')
    _gauge(fig.add_subplot(gs[0, 1]),
           value=row.get('relative_humidity_percent'),
           zones=[(0, 30, '#f4d35e', 'Dry'),
                  (30, 60, '#b7e4c7', 'Comfortable'),
                  (60, 100, '#74c0fc', 'Humid')],
           value_label='Relative humidity (%)',
           title='Humidity at peak',
           xlim=(0, 100), unit_suffix=' %')

    ax = fig.add_subplot(gs[1, :])
    thermal = [
        ('Tile temperature',        tile_temp,                                 '#e03131'),
        ('Heat index (feels-like)', row.get('heat_index_celsius'),             '#f76707'),
        ('Apparent temperature',    row.get('apparent_temperature_celsius'),   '#fd7e14'),
        ('Wet-bulb (evap limit)',   row.get('wet_bulb_temperature_celsius'),   '#20c997'),
    ]
    thermal = [(k, v, c) for k, v, c in thermal if v is not None and v >= 0]
    labels, values, colors = zip(*thermal) if thermal else ([], [], [])
    if values:
        bars = ax.barh(labels, values, color=colors)
        for bar, v in zip(bars, values):
            ax.text(v + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{v} °C', va='center', fontsize=10, fontweight='bold')
        ax.set_xlim(0, max(values) * 1.18)
    ax.set_xlabel('°C')
    ax.set_title(f'Thermal breakdown at peak — {label}')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)

    ax = fig.add_subplot(gs[2, 0])
    aq = {
        'PM2.5':     row.get('air_quality_pm2p5:idx'),
        'PM10':      row.get('air_quality_pm10:idx'),
        'O₃':        row.get('air_quality_o3:idx'),
        'SO₂':       row.get('air_quality_so2:idx'),
        'NO₂':       row.get('air_quality_no2:idx'),
        'CO (AQI)':  row.get('aqi_us_co'),
        'Methane':   row.get('methane_ppb'),
    }
    aq = {k: v for k, v in aq.items() if v is not None and v >= 0}
    if aq:
        keys, vals = list(aq.keys()), list(aq.values())
        ax.barh(keys, vals, color='#845ef7')
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('Index value')
        ax.set_title('Air-quality contributors (higher = worse)')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No air-quality data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    ax = fig.add_subplot(gs[2, 1])
    solar_bars = {
        'GHI (global)':  solar_cs.get('ghi'),
        'DNI (direct)':  solar_cs.get('dni'),
        'DHI (diffuse)': solar_cs.get('dhi'),
    }
    solar_bars = {k: v for k, v in solar_bars.items() if v is not None and v >= 0}
    if solar_bars:
        keys, vals = list(solar_bars.keys()), list(solar_bars.values())
        ax.barh(keys, vals, color=['#f59f00', '#e67700', '#ffd43b'])
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v:.0f}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('W/m²')
        ax.set_title('Solar irradiance (clear sky)')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No solar data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    plt.tight_layout(); plt.show()
    return peak_hi, peak_rh, peak_wb, peak_idx


peak_hi_list, peak_rh_list, peak_wb_list, peak_idx_list = [], [], [], []
env_dfs = {}  # property → DataFrame, kept for downstream inspection if useful

for _, r in top.iterrows():
    env = submit_and_wait_quiet(
        client.environmental_parameters,
        f"env-params #{r['rank']} {r['stop_id']}",
        latitude=r.latitude, longitude=r.longitude,
        temperature=float(r.temperature_c),
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3,                # single day — returns the full 24h diurnal series
    )
    res    = env['result']

    # Persist the raw response.
    out_path = ENV_DIR / f"env_params_{r['stop_id']}_{STUDY_DATE}_live.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f"  saved → {out_path.relative_to(ROOT)}")

    loc    = res['locations'][0]
    params = loc.get('parameters', {}) or {}
    solar_cs = (loc.get('solar_irradiance') or {}).get('clear_sky', {}) or {}
    ts     = pd.to_datetime(res['metadata'].get('timestamps', []))

    env_df = pd.DataFrame({k: v for k, v in params.items()
                           if isinstance(v, list) and len(v) == len(ts)})
    env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)
    env_dfs[r.stop_id] = env_df

    label = f"#{r['rank']} {r['stop_id']}"
    p_hi, p_rh, p_wb, p_idx = _show_env_for_stop(
        env_df, solar_cs, float(r.temperature_c), label,
    )
    peak_hi_list.append(p_hi)
    peak_rh_list.append(p_rh)
    peak_wb_list.append(p_wb)
    peak_idx_list.append(p_idx)

    # Per-stop driver readout.
    print(f"\n{label} — peak hour {p_idx.strftime('%H:%M') if p_idx is not None else '—'}")
    print(f"  Heat index           : {p_hi} °C")
    print(f"  Relative humidity    : {p_rh} %")
    print(f"  Wet-bulb temperature : {p_wb} °C")

top['peak_hi_c']   = peak_hi_list
top['peak_rh_pct'] = peak_rh_list
top['peak_wb_c']   = peak_wb_list

# Scalars for Step 9 (consumes #1's values).
peak_hi = top['peak_hi_c'].iloc[0]
peak_rh = top['peak_rh_pct'].iloc[0]
peak_wb = top['peak_wb_c'].iloc[0]

# --- Map of all stops with their peak driver readout. ---
fmap_env = folium.Map(
    location=[top['latitude'].mean(), top['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{r['rank']} {r['stop_id']} — {r['name']}<br>"
               f"peak HI {r.peak_hi_c} °C · RH {r.peak_rh_pct}%<br>"
               f"wet-bulb {r.peak_wb_c} °C"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_env)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f"~{GRANULARITY_M}m tile",
    ).add_to(fmap_env)

print(f"\nExported to Step 9 (#1 stop) → "
      f"peak_hi={peak_hi}°C, peak_rh={peak_rh}%, peak_wb={peak_wb}°C")

fmap_env

---
## Step 8b — Load cached environmental parameters for a hot tile

### What you are doing
Reading a pre-saved environmental parameters result from `data/env_parameters_urban_planner.json`. This file was produced by running environmental parameters at the centroid of one of the hot tiles identified earlier. We display the coordinates, elevation, the time range covered, the parameter values (heat index, apparent temperature, wet-bulb, humidity, air quality, etc.), the solar irradiance, and the location on a map.

### Why this matters
Use this path when you already have a cached environmental parameters snapshot and want to inspect it without burning API credits. The file contains the full parameter set at the requested time(s) — everything you need to understand the environmental drivers at a hotspot.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import folium

# Cached env-params results live under data/env_params/.
# Step 8 writes new live captures here too — change the filename below to load any captured run.
ENV_DIR  = ROOT / 'data' / 'env_params'
ENV_PATH = ENV_DIR / 'env_parameters_urban_planner.json'
with open(ENV_PATH, 'r') as f:
    env_data = json.load(f)

# --- Location ---
loc_cached = env_data['locations'][0]
lat, lon = float(loc_cached['lat']), float(loc_cached['lon'])
elevation = loc_cached.get('elevation')
tile_temp = loc_cached.get('temperature')
print(f"Location    : ({lat}, {lon})")
print(f"Elevation   : {elevation} m")
print(f"Tile temp   : {tile_temp} °C")

# --- Time range ---
meta = env_data.get('metadata', {})
tr = meta.get('time_range', {})
print(f"Time range  : {tr.get('start')} → {tr.get('end')}  (interval: {tr.get('interval')}, count: {tr.get('count')})")

ts = pd.to_datetime(meta.get('timestamps', []))
params = loc_cached.get('parameters', {})
solar_cs = loc_cached.get('solar_irradiance', {}).get('clear_sky', {})

env_df = pd.DataFrame({k: v for k, v in params.items()
                       if isinstance(v, list) and len(v) == len(ts)})
env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)


def _gauge(ax, value, zones, value_label, title, xlim, unit_suffix):
    """Horizontal gauge: colored zones on bottom 60%, marker + value on top 40%."""
    for lo_z, hi_z, color, label in zones:
        ax.axvspan(lo_z, hi_z, color=color, alpha=0.75, ymin=0, ymax=0.6)
        ax.text((lo_z + hi_z) / 2, 0.3, label, ha='center', va='center', fontsize=8)
    if value is not None and value >= 0:
        ax.plot([value], [0.78], marker='v', markersize=16, color='black', zorder=10)
        ax.text(value, 0.93, f'{value}{unit_suffix}', ha='center', fontsize=10, fontweight='bold')
    ax.set_xlim(*xlim); ax.set_ylim(0, 1); ax.set_yticks([])
    ax.set_xlabel(value_label); ax.set_title(title)


# --- Visualization: rich single-snapshot view, or line plot for time series. ---
if len(env_df) == 1:
    row = env_df.iloc[0]

    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1.1, 1.1], hspace=0.55, wspace=0.28)

    # Row 1: gauges
    _gauge(fig.add_subplot(gs[0, 0]),
           value=row.get('heat_index_celsius'),
           zones=[(15, 27, '#b7e4c7', 'Safe'),
                  (27, 32, '#ffd43b', 'Caution'),
                  (32, 41, '#fd7e14', 'Extreme\ncaution'),
                  (41, 54, '#fa5252', 'Danger')],
           value_label='Heat index (°C) — NWS heat-stress scale',
           title=f"Snapshot at {ts[0].strftime('%Y-%m-%d %H:%M')}",
           xlim=(15, 54), unit_suffix=' °C')
    _gauge(fig.add_subplot(gs[0, 1]),
           value=row.get('relative_humidity_percent'),
           zones=[(0, 30, '#f4d35e', 'Dry'),
                  (30, 60, '#b7e4c7', 'Comfortable'),
                  (60, 100, '#74c0fc', 'Humid')],
           value_label='Relative humidity (%)',
           title='Humidity snapshot',
           xlim=(0, 100), unit_suffix=' %')

    # Row 2: thermal-metrics breakdown — shows what amplifies/dampens the raw tile temp.
    ax = fig.add_subplot(gs[1, :])
    thermal = [
        ('Tile temperature',       tile_temp,                                 '#e03131'),
        ('Heat index (feels-like)', row.get('heat_index_celsius'),            '#f76707'),
        ('Apparent temperature',    row.get('apparent_temperature_celsius'),  '#fd7e14'),
        ('Wet-bulb (evap limit)',   row.get('wet_bulb_temperature_celsius'),  '#20c997'),
    ]
    thermal = [(k, v, c) for k, v, c in thermal if v is not None and v >= 0]
    labels, values, colors = zip(*thermal) if thermal else ([], [], [])
    if values:
        bars = ax.barh(labels, values, color=colors)
        for bar, v in zip(bars, values):
            ax.text(v + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{v} °C', va='center', fontsize=10, fontweight='bold')
        ax.set_xlim(0, max(values) * 1.18)
    ax.set_xlabel('°C')
    ax.set_title('Thermal breakdown — tile temp vs perceived heat (gap reveals humidity contribution)')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)

    # Row 3 left: air-quality contributors (drops −999 sentinels automatically).
    ax = fig.add_subplot(gs[2, 0])
    aq = {
        'PM2.5':     row.get('air_quality_pm2p5:idx'),
        'PM10':      row.get('air_quality_pm10:idx'),
        'O₃':        row.get('air_quality_o3:idx'),
        'SO₂':       row.get('air_quality_so2:idx'),
        'NO₂':       row.get('air_quality_no2:idx'),
        'CO (AQI)':  row.get('aqi_us_co'),
        'Methane':   row.get('methane_ppb'),
    }
    aq = {k: v for k, v in aq.items() if v is not None and v >= 0}
    if aq:
        keys, vals = list(aq.keys()), list(aq.values())
        ax.barh(keys, vals, color='#845ef7')
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('Index value')
        ax.set_title('Air-quality contributors (higher = worse)')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No air-quality data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    # Row 3 right: solar irradiance — direct sun is the heat source to shade against.
    ax = fig.add_subplot(gs[2, 1])
    solar_bars = {
        'GHI (global)':  solar_cs.get('ghi'),
        'DNI (direct)':  solar_cs.get('dni'),
        'DHI (diffuse)': solar_cs.get('dhi'),
    }
    solar_bars = {k: v for k, v in solar_bars.items() if v is not None and v >= 0}
    if solar_bars:
        keys, vals = list(solar_bars.keys()), list(solar_bars.values())
        ax.barh(keys, vals, color=['#f59f00', '#e67700', '#ffd43b'])
        for i, v in enumerate(vals):
            ax.text(v + max(vals) * 0.02, i, f'{v:.0f}', va='center', fontsize=9)
        ax.set_xlim(0, max(vals) * 1.18)
        ax.set_xlabel('W/m²')
        ax.set_title('Solar irradiance (clear sky) — DNI high ⇒ direct sun dominant, shade highly effective')
        ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No solar data', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

    plt.tight_layout(); plt.show()

    # Compact readout + a short driver interpretation.
    hi, app, wb, rh = (row.get('heat_index_celsius'),
                       row.get('apparent_temperature_celsius'),
                       row.get('wet_bulb_temperature_celsius'),
                       row.get('relative_humidity_percent'))
    print(f"\nThermal readout at this hour:")
    print(f"  Heat index           : {hi} °C")
    print(f"  Apparent temperature : {app} °C")
    print(f"  Wet-bulb temperature : {wb} °C")
    print(f"  Relative humidity    : {rh} %")
    if hi is not None and tile_temp is not None:
        gap = tile_temp - hi
        print(f"\nDriver interpretation:")
        if rh is not None and rh < 40:
            print(f"  • Dry air (RH {rh}%) → sweating is effective; evaporative cooling (misting) will work well here.")
        elif rh is not None and rh >= 60:
            print(f"  • Humid air (RH {rh}%) → sweating is suppressed; ventilation/shade beats misting.")
        if wb is not None and wb > 28:
            print(f"  • Wet-bulb {wb} °C approaches human limits — prolonged outdoor waiting is unsafe.")
        print(f"  • Solar load DNI={solar_cs.get('dni', '?')} W/m² is the primary heat input — full shade blocks ~70–90% of this.")
else:
    # Multi-timestamp fallback: line plot through the day.
    plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                             'wet_bulb_temperature_celsius', 'relative_humidity_percent')
                 if c in env_df.columns and (env_df[c] >= 0).all()]
    if plot_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        thermal_cols = [c for c in plot_cols if 'humidity' not in c]
        if thermal_cols:
            env_df[thermal_cols].plot(ax=axes[0], marker='o')
            axes[0].set_title(f"Thermal comfort at ({lat:.4f}, {lon:.4f})")
            axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
        if 'relative_humidity_percent' in env_df.columns:
            env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
            axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

# --- Location on map ---
fmap_env = folium.Map(location=[lat, lon], zoom_start=16, tiles='cartodbpositron')
folium.Marker(
    location=[lat, lon],
    popup=f"Env params<br>({lat:.6f}, {lon:.6f})<br>{tile_temp} °C",
    icon=folium.Icon(color='red', icon='info-sign'),
).add_to(fmap_env)
folium.Circle(
    location=[lat, lon], radius=GRANULARITY_M / 2,
    color='red', fill=True, fill_opacity=0.15,
    popup=f"~{GRANULARITY_M}m tile",
).add_to(fmap_env)

# --- Expose driver values for Step 9 (cached-path compatibility). ---
peak_hi = env_df['heat_index_celsius'].max() if 'heat_index_celsius' in env_df.columns else None
if 'heat_index_celsius' in env_df.columns:
    peak_idx = env_df['heat_index_celsius'].idxmax()
    peak_rh = env_df.loc[peak_idx, 'relative_humidity_percent'] if 'relative_humidity_percent' in env_df.columns else None
    peak_wb = env_df.loc[peak_idx, 'wet_bulb_temperature_celsius'] if 'wet_bulb_temperature_celsius' in env_df.columns else None
else:
    peak_rh = peak_wb = None
print(f"\nExported to Step 9 → peak_hi={peak_hi}°C, peak_rh={peak_rh}%, peak_wb={peak_wb}°C")

fmap_env

---
## Step 9 — Prioritized action list

### What you are doing
Combining everything we learned into a single action-oriented DataFrame. For each of the top stops, a **scoring heuristic** weighs several candidate interventions and returns the top two as `primary_action` and `secondary_action`. The scoring rules:

| Rule | When it fires | What it scores |
|---|---|---|
| **Wet-bulb safety** | `wet_bulb > 30 °C` at peak | Always wins — closes stop until retrofit |
| **Full-coverage canopy** | #1 stop AND `sky_pct > 45%` | Priority scales with temperature delta vs AOI mean |
| **Partial canopy** | `temp delta > 2 °C` AND `veg < 20%` | Priority scales with delta |
| **Street trees** | `vegetation < 15%` | Scales with how far below target + delta |
| **Cool-surface paving** | `impervious > 60%` | Scales with impervious excess + delta |
| **Misting station** | `peak heat index > 30 °C` AND `RH < 40%` | Dry regime — evaporative cooling works |
| **Ventilated shelter** | `peak heat index > 30 °C` AND `RH ≥ 60%` | Humid regime — misting ineffective |
| **Hybrid cooling** | `peak heat index > 30 °C`, moderate RH | Compromise |

### Why this matters
This is the row the council actually reads. Two things make the output defensible:

1. **Humidity picks the intervention type.** The same heat index + dry air calls for misting; the same heat index + humid air calls for ventilation. Past dashboards that ignored humidity frequently recommended the wrong fix.
2. **Stops differentiate on temperature delta, humidity regime, and rank** — not just raw surface percentages. Even when two stops share similar satellite/street-view numbers, their scored primary actions can differ because `delta` and the rank-gated canopy rule shift the ranking.

Every column is traceable:
- `temperature_c` — Step 3 (spatial join)
- `impervious_pct` / `vegetation_pct` — Step 6 (or 6b)
- `primary_action` — derived here, driven by scores combining Steps 3, 6, 7, 8
- `primary_reason` / `secondary_reason` — the exact thresholds that fired, so you can defend each recommendation.

In [ ]:
aoi_mean_t = t_stats.get('mean')
# env-driven peaks — fall through to None if Step 8/8b didn't run.
_peak_rh = globals().get('peak_rh')
_peak_wb = globals().get('peak_wb')


def _recommend(row, *, sky_pct_top1, peak_hi, peak_rh, peak_wb, aoi_mean_t, is_top1):
    """Score candidate interventions, return primary + secondary with rationale."""
    temp  = row.get('temperature_c') or 0
    veg   = row.get('vegetation_pct') if row.get('vegetation_pct') is not None else 0
    imp   = row.get('impervious_pct') if row.get('impervious_pct') is not None else 0
    delta = (temp - aoi_mean_t) if aoi_mean_t is not None else 0

    candidates = []  # (score, action, reason)

    # Wet-bulb safety gate — always wins when triggered.
    if peak_wb is not None and peak_wb > 30:
        candidates.append((20,
            'URGENT: close stop during peak hours until retrofit complete',
            f'wet-bulb {peak_wb} °C — above survivable threshold for prolonged waiting'))

    # Full shade canopy: #1 stop with open sky, OR any stop running hot with low veg.
    if is_top1 and sky_pct_top1 is not None and sky_pct_top1 > 45:
        candidates.append((12 + delta,
            'Install full-coverage shade canopy over shelter',
            f'open-sky {sky_pct_top1}% at hottest stop; tile {temp} °C ({delta:+.1f} vs AOI mean)'))
    elif delta > 2.0 and veg < 20:
        candidates.append((6 + delta,
            'Install partial shade canopy over seating area',
            f'tile {temp} °C ({delta:+.1f} vs mean) with tree cover only {veg}%'))

    # Street trees — mid-term cooling; gated on low vegetation.
    if veg < 15:
        candidates.append((3 + (15 - veg) * 0.3 + delta * 0.4,
            'Plant street trees along pedestrian approach (3–5 yr cooling payoff)',
            f'vegetation {veg}% (target ≥ 15%)'))

    # Reflective / cool-surface paving — gated on high impervious + hot tile.
    if imp > 60:
        candidates.append((4 + (imp - 60) * 0.08 + delta * 0.5,
            'Apply reflective / cool-surface paving around shelter footprint',
            f'impervious {imp}% absorbs solar radiation; tile {delta:+.1f} °C vs mean'))

    # Active cooling — type chosen by humidity regime.
    if peak_hi is not None and peak_hi > 30:
        if peak_rh is not None and peak_rh < 40:
            candidates.append((6 + (peak_hi - 30) + delta * 0.3,
                'Install misting station (evaporative cooling — effective in dry air)',
                f'heat index {peak_hi} °C with RH {peak_rh}% (dry regime)'))
        elif peak_rh is not None and peak_rh >= 60:
            candidates.append((6 + (peak_hi - 30) + delta * 0.3,
                'Retrofit shelter: cross-ventilation + reflective roof (humid regime — misting ineffective)',
                f'heat index {peak_hi} °C with RH {peak_rh}% (humid regime)'))
        else:
            candidates.append((5 + (peak_hi - 30) + delta * 0.3,
                'Hybrid cooling: ventilated shelter + low-volume misting',
                f'heat index {peak_hi} °C with RH {peak_rh}% (moderate humidity)'))

    # Default when nothing triggered.
    if not candidates:
        candidates.append((1,
            'Monitor — below intervention thresholds',
            f'tile {temp} °C; no individual driver exceeds threshold'))

    candidates.sort(key=lambda c: -c[0])
    primary = candidates[0]
    secondary = next((c for c in candidates[1:] if c[1] != primary[1]), None)

    return pd.Series({
        'primary_action':    primary[1],
        'primary_reason':    primary[2],
        'secondary_action':  secondary[1] if secondary else '—',
        'secondary_reason':  secondary[2] if secondary else '—',
    })


actions = top.apply(
    lambda r: _recommend(r,
                         sky_pct_top1=sky_pct if r['rank'] == 1 else None,
                         peak_hi=peak_hi, peak_rh=_peak_rh, peak_wb=_peak_wb,
                         aoi_mean_t=aoi_mean_t,
                         is_top1=r['rank'] == 1),
    axis=1,
)
action_list = pd.concat([top.reset_index(drop=True),
                         actions.reset_index(drop=True)], axis=1)

display_cols = ['rank', 'stop_id', 'name', 'temperature_c',
                'impervious_pct', 'vegetation_pct',
                'primary_action', 'primary_reason',
                'secondary_action', 'secondary_reason']
action_list[display_cols]

In [ ]:
# Build the output bundle root and save the action list CSV here.
# (Step 10 below adds the PDF report and per-flow folium maps to the same folder.)
OUT_DIR = ROOT / 'outputs' / f'bus_stops_{STUDY_DATE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / 'action_list.csv'
action_list.to_csv(csv_path, index=False)
print(f'Saved prioritized action list to {csv_path.relative_to(ROOT)}')

---
## Step 10 — Package outputs (CSV + PDF + maps)

### What you are doing
Bundling everything the analysis just produced into a single hand-off folder under `outputs/bus_stops_{STUDY_DATE}/`:

- `action_list.csv` — the prioritized action list from Step 9 (already saved above).
- `bus_stops_report.pdf` — multi-page slide-deck-ready PDF with the heatmap summary, the top-N satellite/street/env diagnoses, and the action table.
- `maps/*.html` — every interactive folium map (overall heatmap, hot-stops cluster, top-N satellite, top-N street view, top-N env-params) saved as standalone HTML, openable in any browser.

### Why this matters
Different stakeholders consume different formats. The operations team wants a CSV they can open in Excel; the council wants a PDF for the agenda packet; the design team wants the interactive maps to zoom and click around. This step produces all three at once, named consistently, in one folder you can hand off.

In [ ]:
import io, base64, json, shutil
from PIL import Image as PILImage

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle,
        PageBreak, CondPageBreak, KeepTogether, Flowable,
    )
except ImportError:
    raise RuntimeError(
        "reportlab is required for the PDF export. "
        "Install with: pip install 'reportlab>=4.0.0' (or pip install -r requirements.txt)."
    )

OUT_DIR = ROOT / 'outputs' / f'bus_stops_{STUDY_DATE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'maps').mkdir(exist_ok=True)


# --- 1. Save folium maps as standalone HTML. --------------------------------
maps = [
    ('overall_heatmap.html',     globals().get('fmap')),
    ('hot_stops_cluster.html',   globals().get('fmap_hot')),
    ('satellite_top_n.html',     globals().get('fmap_sat') or globals().get('fmap_seg')),
    ('street_view_top_n.html',   globals().get('fmap_street')),
    ('env_params_top_n.html',    globals().get('fmap_env')),
]
for fname, m in maps:
    if m is None:
        continue
    m.save(str(OUT_DIR / 'maps' / fname))
    print(f'  ✓ maps/{fname}')


# --- 2. PDF report (ReportLab Platypus). ------------------------------------
PAGE_W, PAGE_H = letter
LEFT_MARGIN  = 0.6 * inch
RIGHT_MARGIN = 0.6 * inch
TOP_MARGIN   = 0.7 * inch
BOT_MARGIN   = 0.7 * inch
USABLE_W     = PAGE_W - LEFT_MARGIN - RIGHT_MARGIN

# Brand palette (FortyGuard).
BRAND_BLUE   = colors.HexColor('#0E4A8A')
BRAND_INK    = colors.HexColor('#1f2933')
BRAND_MUTED  = colors.HexColor('#5a6b7b')
BRAND_YELLOW = colors.HexColor('#FFD24D')

LOGO_PATH        = ROOT / 'assets' / 'fortyguard_logo.png'           # white wordmark — for blue backgrounds
LOGO_FOOTER_PATH = ROOT / 'assets' / 'fortyguard_logo_footer.png'    # two-tone blue — for white backgrounds
COVER_BG_PATH    = ROOT / 'assets' / 'cover_bg.png'                  # full-bleed cover background
LOGO_ASPECT        = 224 / 1208   # h / w of the cover wordmark
LOGO_FOOTER_ASPECT = 68 / 364     # h / w of the footer wordmark

_styles = getSampleStyleSheet()
S_H1    = ParagraphStyle('H1', parent=_styles['Heading1'],
                         fontName='Helvetica-Bold', fontSize=16, leading=20,
                         textColor=BRAND_BLUE, spaceBefore=4, spaceAfter=10)
S_H2    = ParagraphStyle('H2', parent=_styles['Heading2'],
                         fontName='Helvetica-Bold', fontSize=12, leading=15,
                         textColor=BRAND_INK, spaceBefore=4, spaceAfter=6)
S_BODY  = ParagraphStyle('Body', parent=_styles['BodyText'],
                         fontName='Helvetica', fontSize=10, leading=14,
                         textColor=BRAND_INK, spaceAfter=4)
S_BODY_W = ParagraphStyle('BodyW', parent=S_BODY, fontName='Helvetica', fontSize=8, leading=11)
S_CAP   = ParagraphStyle('Cap', parent=_styles['Italic'],
                         fontName='Helvetica-Oblique', fontSize=9, leading=12,
                         textColor=BRAND_MUTED, spaceAfter=8)
S_CONTACT = ParagraphStyle('Contact', parent=_styles['Normal'],
                           fontName='Helvetica', fontSize=9, leading=13,
                           textColor=BRAND_INK, spaceAfter=4)


def _h1(text):
    """Section heading: uppercase + brand-blue (A4 design language)."""
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H1)


def _h2(text):
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H2)


def _fig_to_image(fig, max_width_inches=6.6, dpi=180):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    img = PILImage.open(buf)
    iw, ih = img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _pil_to_image(pil_img, max_width_inches=6.6):
    if pil_img is None:
        return None
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    buf.seek(0)
    iw, ih = pil_img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _decode_b64(b64_str):
    if not b64_str:
        return None
    if isinstance(b64_str, list):
        b64_str = b64_str[0]
    if b64_str.startswith('data:'):
        b64_str = b64_str.split(',', 1)[1]
    try:
        return PILImage.open(io.BytesIO(base64.b64decode(b64_str)))
    except Exception:
        return None


def _wrap(s, style=S_BODY):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return Paragraph('—', style)
    return Paragraph(str(s).replace('\n', '<br/>'), style)


TABLE_STYLE = TableStyle([
    ('BACKGROUND',    (0, 0),  (-1, 0),   BRAND_BLUE),
    ('TEXTCOLOR',     (0, 0),  (-1, 0),   colors.white),
    ('FONTNAME',      (0, 0),  (-1, 0),   'Helvetica-Bold'),
    ('FONTSIZE',      (0, 0),  (-1, 0),   9),
    ('ALIGN',         (0, 0),  (-1, 0),   'LEFT'),
    ('VALIGN',        (0, 0),  (-1, -1),  'MIDDLE'),
    ('LINEBELOW',     (0, 0),  (-1, 0),   0.6, BRAND_BLUE),
    ('LINEBELOW',     (0, -1), (-1, -1),  0.4, colors.HexColor('#cbd5e0')),
    ('FONTSIZE',      (0, 1),  (-1, -1),  8),
    ('LEFTPADDING',   (0, 0),  (-1, -1),  6),
    ('RIGHTPADDING',  (0, 0),  (-1, -1),  6),
    ('TOPPADDING',    (0, 0),  (-1, -1),  5),
    ('BOTTOMPADDING', (0, 0),  (-1, -1),  5),
    ('ROWBACKGROUNDS',(0, 1),  (-1, -1),  [colors.white, colors.HexColor('#f4f7fb')]),
])


def _draw_cover(canvas, doc):
    """Page 1: full-bleed bg.png, yellow accents, white wordmark at lower-left."""
    canvas.saveState()

    # 1. Full-page background image (or solid blue fallback).
    if COVER_BG_PATH.exists():
        canvas.drawImage(str(COVER_BG_PATH), 0, 0,
                         width=PAGE_W, height=PAGE_H,
                         preserveAspectRatio=False, mask='auto')
    else:
        canvas.setFillColor(BRAND_BLUE)
        canvas.rect(0, 0, PAGE_W, PAGE_H, stroke=0, fill=1)

    # 2. Yellow pill containing the study date, top-left.
    pill_h = 0.34 * inch
    pill_w = 1.45 * inch
    pill_x = LEFT_MARGIN
    pill_y = PAGE_H - 1.05 * inch
    canvas.setFillColor(BRAND_YELLOW)
    canvas.roundRect(pill_x, pill_y, pill_w, pill_h, pill_h / 2,
                     stroke=0, fill=1)
    canvas.setFillColor(BRAND_INK)
    canvas.setFont('Helvetica-Bold', 11)
    canvas.drawCentredString(pill_x + pill_w / 2,
                             pill_y + pill_h / 2 - 0.04 * inch,
                             STUDY_DATE)

    # 3. Title — large white uppercase, wraps onto 3 short lines.
    canvas.setFillColor(colors.white)
    canvas.setFont('Helvetica-Bold', 38)
    title_top_y = PAGE_H - 1.8 * inch
    line_h      = 0.55 * inch
    canvas.drawString(LEFT_MARGIN, title_top_y - 0 * line_h, 'BUS-STOP COOLING')
    canvas.drawString(LEFT_MARGIN, title_top_y - 1 * line_h, 'INTERVENTION')
    canvas.drawString(LEFT_MARGIN, title_top_y - 2 * line_h, 'REPORT')

    # 4. Info lines — yellow label + white value.
    def _info_line(y, label, value):
        canvas.setFont('Helvetica-Bold', 11)
        canvas.setFillColor(BRAND_YELLOW)
        canvas.drawString(LEFT_MARGIN, y, label)
        label_w = canvas.stringWidth(label, 'Helvetica-Bold', 11)
        canvas.setFont('Helvetica', 11)
        canvas.setFillColor(colors.white)
        canvas.drawString(LEFT_MARGIN + label_w + 0.06 * inch, y, value)

    info_y = title_top_y - 2 * line_h - 0.55 * inch
    _info_line(info_y, 'AOI:',
               f' {len(features):,} tiles at {GRANULARITY_M} m  ·  '
               f'{len(ranked)} stops  ·  top {TOP_N_TO_DIAGNOSE} diagnosed in detail')

    aoi_min = t_stats.get('min'); aoi_mean = t_stats.get('mean'); aoi_max = t_stats.get('max')
    info2_y = info_y - 0.28 * inch
    if aoi_min is not None:
        _info_line(info2_y, 'AOI peak temperature range:',
                   f' {aoi_min:.1f} – {aoi_max:.1f} °C  (mean {aoi_mean:.1f} °C)')

    # 5. Three yellow dots beneath the info lines.
    upper_dots_y = (info2_y if aoi_min is not None else info_y) - 0.40 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, upper_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    # 6. Bottom block — three yellow dots + white wordmark, left-aligned.
    logo_w = 3.6 * inch
    logo_h = logo_w * LOGO_ASPECT
    logo_y = 1.2 * inch
    if LOGO_PATH.exists():
        canvas.drawImage(str(LOGO_PATH), LEFT_MARGIN, logo_y,
                         width=logo_w, height=logo_h, mask='auto')

    lower_dots_y = logo_y + logo_h + 0.28 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, lower_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    canvas.restoreState()


def _draw_body(canvas, doc):
    """Pages 2+: page number at bottom-right; nothing else."""
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(BRAND_MUTED)
    canvas.drawRightString(PAGE_W - RIGHT_MARGIN, 0.4 * inch, f'Page {doc.page}')
    canvas.restoreState()

class _BottomAnchored(Flowable):
    """Consume remaining frame height and anchor child flowables to the bottom edge."""
    def __init__(self, children):
        Flowable.__init__(self)
        self.children = children
        self._sizes = []

    def wrap(self, aW, aH):
        self.width = aW
        self.height = aH
        self._sizes = [c.wrap(aW, aH) for c in self.children]
        return (aW, aH)

    def draw(self):
        total = sum(h for _, h in self._sizes)
        y = total
        for c, (cw, ch) in zip(self.children, self._sizes):
            y -= ch
            ha = getattr(c, 'hAlign', 'LEFT')
            if ha in ('CENTER', 'CENTRE', 1):
                x = (self.width - cw) / 2
            elif ha in ('RIGHT', 2):
                x = self.width - cw
            else:
                x = 0
            c.drawOn(self.canv, x, y)


# Matplotlib figure builders.
def _build_heatmap_distribution_fig():
    temps = [f['properties']['temperature'] for f in features
             if f['properties'].get('temperature') is not None]
    if not temps:
        return None
    lo, hi, mean = float(min(temps)), float(max(temps)), float(sum(temps) / len(temps))

    fig = plt.figure(figsize=(11, 4.5), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    # Rounded white card behind the legend column.
    from matplotlib.patches import FancyBboxPatch
    ax0.add_patch(FancyBboxPatch(
        (-0.02, -0.02), 1.04, 1.04,
        transform=ax0.transAxes,
        boxstyle='round,pad=0.02,rounding_size=0.04',
        facecolor='white', edgecolor='#d8dee6', linewidth=0.8,
        clip_on=False, zorder=0,
    ))
    ax0.text(0.05, 0.92, f'Heatmap · {STUDY_DATE}', transform=ax0.transAxes,
             fontsize=12, fontweight='bold', va='top', zorder=2)
    ax0.text(0.05, 0.82, f'{len(temps):,} tiles', transform=ax0.transAxes,
             fontsize=10, color='#666', va='top', zorder=2)
    rows = [('min', lo), ('mean', mean), ('max', hi)]
    y = 0.62
    for label, val in rows:
        ax0.text(0.05, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace', zorder=2)
        ax0.add_patch(plt.Rectangle((0.27, y - 0.06), 0.10, 0.12,
                                    transform=ax0.transAxes,
                                    facecolor=temp_color(val, lo, hi),
                                    edgecolor='#333', linewidth=0.6, zorder=2))
        ax0.text(0.42, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=12, fontweight='bold', va='center', family='monospace', zorder=2)
        y -= 0.18

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, e_lo, e_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((e_lo + e_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile peak temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Daily peak temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for sp in ('top', 'right'):
        ax1.spines[sp].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP, extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([]); ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)
    return fig


def _build_seg_breakdown_fig(segments, title, color='#3b8686'):
    if not segments:
        return None
    items   = sorted(segments.items(), key=lambda x: float(x[1]), reverse=True)
    classes = [k for k, _ in items]
    pcts    = [float(v) for _, v in items]
    fig, ax = plt.subplots(figsize=(8.5, max(2.5, 0.4 + 0.35 * len(classes))))
    ax.barh(classes, pcts, color=color, edgecolor='#333', linewidth=0.6)
    for i, p in enumerate(pcts):
        ax.text(p + max(pcts) * 0.015, i, f'{p:.1f}%',
                va='center', fontsize=9, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlabel('Coverage (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def _build_top_n_surface_comp_fig(seg_dict, title):
    rows = [(pid, segs) for pid, segs in (seg_dict or {}).items() if segs]
    if not rows:
        return None
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of scene')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    return fig


def _build_diurnal_fig(env_df, title):
    if env_df is None or not len(env_df) or 'heat_index_celsius' not in env_df.columns:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                             'wet_bulb_temperature_celsius') if c in env_df.columns]
    if plot_cols:
        env_df[plot_cols].plot(ax=axes[0], marker='o')
        axes[0].set_title(title, fontsize=11, fontweight='bold')
        axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
    if 'relative_humidity_percent' in env_df.columns:
        env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
        axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    return fig


# Build flowables.
flowables = []

# Cover: drawn entirely on the canvas in _draw_cover. PageBreak() advances
# past page 1 so body content starts on page 2.
flowables.append(PageBreak())

# Heatmap distribution
flowables.append(_h1('AOI peak-temperature distribution'))
flowables.append(Paragraph(
    'Per-tile daily peak across the AOI. The dashed line marks the AOI mean; '
    'the histogram is colored on the same spectral ramp as every map and chart in this report.',
    S_BODY))
hm_fig = _build_heatmap_distribution_fig()
if hm_fig is not None:
    flowables.append(Spacer(1, 0.1 * inch))
    flowables.append(_fig_to_image(hm_fig, max_width_inches=USABLE_W / inch))
flowables.append(Spacer(1, 0.35 * inch))
flowables.append(CondPageBreak(3.5 * inch))

# Top-N at a glance
flowables.append(_h1(f'Top {TOP_N_TO_DIAGNOSE} hottest stops — at a glance'))
flowables.append(Paragraph(
    'Daily peak temperature, surface diagnosis (% impervious vs. vegetation), '
    'sky exposure, and peak heat index for each of the top-N stops.',
    S_BODY))

top_cols = ['Rank', 'Stop ID', 'Name', 'Peak °C',
            'Imperv %', 'Veg %', 'Sky %', 'HI °C', 'RH %', 'Wet-bulb °C']
top_rows = []
for _, r in top.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    top_rows.append([
        f'#{int(r["rank"])}',
        r.stop_id,
        _wrap(r['name'], S_BODY),
        _f(r.temperature_c),
        _f(r.get('impervious_pct'), '{:.0f}'),
        _f(r.get('vegetation_pct'), '{:.0f}'),
        _f(r.get('sky_pct'), '{:.0f}'),
        _f(r.get('peak_hi_c')),
        _f(r.get('peak_rh_pct'), '{:.0f}'),
        _f(r.get('peak_wb_c')),
    ])
top_table = Table([top_cols] + top_rows, repeatRows=1, hAlign='LEFT',
                  colWidths=[0.4*inch, 0.6*inch, 1.5*inch, 0.55*inch,
                             0.55*inch, 0.5*inch, 0.5*inch, 0.55*inch, 0.5*inch, 0.7*inch])
top_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(top_table)
flowables.append(Spacer(1, 0.4 * inch))
flowables.append(CondPageBreak(4 * inch))

# Surface composition (C2 / C2b)
sat_seg_dict = {}
sv_seg_dict  = {}
for idx, (_, r) in enumerate(top.iterrows()):
    pid = r.stop_id
    if 'raw_segments' in dir() and idx < len(raw_segments):
        sat_seg_dict[pid] = raw_segments[idx]
    if 'sv_results' in dir() and idx < len(sv_results):
        front = sv_results[idx]
        if isinstance(front, dict):
            sv_seg_dict[pid] = front.get('segments', {}) or {}

if sat_seg_dict or sv_seg_dict:
    flowables.append(_h1('Surface composition across top-N (C2 / C2b)'))
    flowables.append(Paragraph(
        'Stacked bars show the per-stop class breakdown for satellite (overhead) '
        'and street view (front-of-stop). Together they pin down whether the heat '
        'driver is impervious surface, low canopy, or open sky.',
        S_BODY))
    if sat_seg_dict:
        c2_fig = _build_top_n_surface_comp_fig(sat_seg_dict,
                                                'C2 — Satellite surface composition (top-N)')
        if c2_fig is not None:
            flowables.append(Spacer(1, 0.1 * inch))
            flowables.append(_fig_to_image(c2_fig, max_width_inches=USABLE_W / inch))
    if sv_seg_dict:
        c2b_fig = _build_top_n_surface_comp_fig(sv_seg_dict,
                                                 'C2b — Street-view scene composition (top-N)')
        if c2b_fig is not None:
            flowables.append(Spacer(1, 0.2 * inch))
            flowables.append(_fig_to_image(c2b_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.4 * inch))

# Per-stop deep dive — each stop starts a fresh page, but its three
# sub-sections (overview, street-view, env-params) flow continuously.
for idx, (_, r) in enumerate(top.iterrows()):
    pid = r.stop_id
    flowables.append(PageBreak())
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']}"))

    metric_lines = [
        f"<b>Peak temp:</b> {r.temperature_c:.1f} °C  &nbsp;·&nbsp; "
        f"<b>Lat/Lon:</b> {r.latitude:.4f}, {r.longitude:.4f}",
    ]
    if pd.notna(r.get('impervious_pct')) or pd.notna(r.get('vegetation_pct')):
        metric_lines.append(
            f"<b>Surface:</b> impervious "
            f"{r.impervious_pct if pd.notna(r.impervious_pct) else '—'}% &nbsp;·&nbsp; "
            f"vegetation {r.vegetation_pct if pd.notna(r.vegetation_pct) else '—'}%"
        )
    if pd.notna(r.get('sky_pct')) or pd.notna(r.get('building_pct')):
        metric_lines.append(
            f"<b>Street view:</b> sky "
            f"{r.sky_pct if pd.notna(r.sky_pct) else '—'}% &nbsp;·&nbsp; "
            f"building {r.building_pct if pd.notna(r.building_pct) else '—'}%"
        )
    if pd.notna(r.get('peak_hi_c')):
        metric_lines.append(
            f"<b>Peak heat index:</b> {r.peak_hi_c:.1f} °C &nbsp;·&nbsp; "
            f"<b>RH at peak:</b> {r.peak_rh_pct if pd.notna(r.peak_rh_pct) else '—'}% &nbsp;·&nbsp; "
            f"<b>Wet-bulb:</b> {r.peak_wb_c if pd.notna(r.peak_wb_c) else '—'} °C"
        )
    for line in metric_lines:
        flowables.append(Paragraph(line, S_BODY))
    flowables.append(Spacer(1, 0.1 * inch))

    # Satellite imagery
    if 'sat_results' in dir() and idx < len(sat_results):
        res = sat_results[idx]
        seg = res.get('segmentation', {}) or {}
        orig_pil = _decode_b64(res.get('orignal_image') or res.get('original_image'))
        seg_pil  = _decode_b64(seg.get('image_content'))
        if orig_pil is not None or seg_pil is not None:
            flowables.append(_h2('Satellite — original & segmentation'))
            cells = []
            if orig_pil is not None:
                cells.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                              Paragraph('original satellite tile', S_CAP)])
            else:
                cells.append([Paragraph('(original missing)', S_CAP)])
            if seg_pil is not None:
                cells.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                              Paragraph('segmentation overlay', S_CAP)])
            else:
                cells.append([Paragraph('(segmentation missing)', S_CAP)])
            sat_row = Table([[c[0] for c in cells], [c[1] for c in cells]],
                            colWidths=[3.3 * inch, 3.3 * inch])
            sat_row.setStyle(TableStyle([
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                ('LEFTPADDING', (0, 0), (-1, -1), 0),
                ('RIGHTPADDING',(0, 0), (-1, -1), 0),
                ('TOPPADDING',  (0, 0), (-1, -1), 0),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
            ]))
            flowables.append(sat_row)
            flowables.append(Spacer(1, 0.1 * inch))
        if pid in sat_seg_dict and sat_seg_dict[pid]:
            cb_fig = _build_seg_breakdown_fig(sat_seg_dict[pid],
                                               f'Satellite class breakdown — {pid}',
                                               color='#3b8686')
            if cb_fig is not None:
                flowables.append(_fig_to_image(cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(4 * inch))

    # Street view — side-by-side pair, matching the satellite section.
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']} (street view)"))
    if 'sv_results' in dir() and idx < len(sv_results):
        front = sv_results[idx]
        if isinstance(front, dict):
            orig_pil = _decode_b64(front.get('original_image'))
            seg_pil  = _decode_b64(front.get('segmented_image'))
            segs     = front.get('segments', {}) or {}
            if orig_pil is not None or seg_pil is not None:
                cells = []
                if orig_pil is not None:
                    cells.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                                  Paragraph('original street-view (front)', S_CAP)])
                else:
                    cells.append([Paragraph('(original missing)', S_CAP)])
                if seg_pil is not None:
                    cells.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                                  Paragraph('pixel-wise segmentation', S_CAP)])
                else:
                    cells.append([Paragraph('(segmentation missing)', S_CAP)])
                sv_row = Table([[c[0] for c in cells], [c[1] for c in cells]],
                               colWidths=[3.3 * inch, 3.3 * inch])
                sv_row.setStyle(TableStyle([
                    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                    ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                    ('LEFTPADDING',  (0, 0), (-1, -1), 0),
                    ('RIGHTPADDING', (0, 0), (-1, -1), 0),
                    ('TOPPADDING',   (0, 0), (-1, -1), 0),
                    ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
                ]))
                flowables.append(sv_row)
                flowables.append(Spacer(1, 0.1 * inch))
            if segs:
                sv_cb_fig = _build_seg_breakdown_fig(segs,
                                                      f'Street-view class breakdown — {pid}',
                                                      color='#7a5195')
                if sv_cb_fig is not None:
                    flowables.append(_fig_to_image(sv_cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(3.5 * inch))

    # Env-params diurnal
    flowables.append(_h1(f"#{int(r['rank'])} · {pid} — {r['name']} (env-params)"))
    env_df = (env_dfs or {}).get(pid)
    if env_df is not None and len(env_df):
        flowables.append(Paragraph(
            'Diurnal heat-index, apparent temperature, wet-bulb, and RH curves across the day.',
            S_BODY))
        ev_fig = _build_diurnal_fig(env_df, f"Diurnal drivers — {pid}")
        if ev_fig is not None:
            flowables.append(_fig_to_image(ev_fig, max_width_inches=USABLE_W / inch))
    else:
        flowables.append(Paragraph('No env-params data available for this stop.', S_BODY))

# Action list table
flowables.append(PageBreak())
flowables.append(_h1('Prioritized action list'))
flowables.append(Paragraph(
    'Threshold-triggered intervention recommendations from Step 9. The primary action is '
    'the highest-leverage measure given each stop\'s measurements; the secondary is the '
    'next most relevant.',
    S_BODY))

action_cols = ['Rank', 'Stop ID', 'Name', 'Peak °C', 'Primary action', 'Secondary action']
action_rows = []
display_src = action_list if 'action_list' in dir() else top
for _, p in display_src.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    action_rows.append([
        int(p['rank']) if pd.notna(p.get('rank')) else '—',
        p.get('stop_id', '—'),
        _wrap(p.get('name'), S_BODY_W),
        _f(p.get('temperature_c')),
        _wrap(p.get('primary_action', '—'), S_BODY_W),
        _wrap(p.get('secondary_action', '—'), S_BODY_W),
    ])
action_table = Table([action_cols] + action_rows, repeatRows=1, hAlign='LEFT',
                     colWidths=[0.4*inch, 0.6*inch, 1.2*inch, 0.6*inch, 2.3*inch, 2.2*inch])
action_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(action_table)

# Closing contact block — yellow rule, contact heading + text, centered blue wordmark.
# Anchored to the bottom of the current page via _BottomAnchored so the
# block always sits flush with the bottom margin.
yellow_rule = Table([['']], colWidths=[USABLE_W], rowHeights=[0.06 * inch])
yellow_rule.setStyle(TableStyle([
    ('BACKGROUND',    (0, 0), (-1, -1), BRAND_YELLOW),
    ('LEFTPADDING',   (0, 0), (-1, -1), 0),
    ('RIGHTPADDING',  (0, 0), (-1, -1), 0),
    ('TOPPADDING',    (0, 0), (-1, -1), 0),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 0),
]))

closing_block = [
    yellow_rule,
    Spacer(1, 0.25 * inch),
    _h1('FortyGuard contact details'),
    Paragraph(
        'FortyGuard Tech Limited, Al Khatem Tower, 14th Floor, 112, WeWork Hub71, '
        'Abu Dhabi Global Market Square, Al Maryah Island, Abu Dhabi, UAE. '
        'PoBox: 3317, Tel: +97126662799, Email: info@fortyguard.com',
        S_CONTACT),
    Spacer(1, 0.35 * inch),
]

if LOGO_FOOTER_PATH.exists():
    closing_logo_w = 1.8 * inch
    closing_logo_h = closing_logo_w * LOGO_FOOTER_ASPECT
    closing_logo   = RLImage(str(LOGO_FOOTER_PATH),
                             width=closing_logo_w, height=closing_logo_h)
    closing_logo.hAlign = 'CENTER'
    closing_block.append(closing_logo)

flowables.append(_BottomAnchored(closing_block))


# Render the PDF.
pdf_path = OUT_DIR / 'bus_stops_report.pdf'
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=letter,
    leftMargin=LEFT_MARGIN, rightMargin=RIGHT_MARGIN,
    topMargin=TOP_MARGIN,   bottomMargin=BOT_MARGIN,
    title=f'Bus-Stop Cooling Intervention Report · {STUDY_DATE}',
    author='FortyGuard',
)
doc.build(flowables, onFirstPage=_draw_cover, onLaterPages=_draw_body)


# Save action_list CSV in the same bundle (Step 9 also wrote it to action_list.csv).
csv_path = OUT_DIR / 'action_list.csv'
if 'action_list' in dir() and not action_list.empty:
    action_list.to_csv(csv_path, index=False)
    print(f'  ✓ {csv_path.relative_to(ROOT)}')

print(f'  ✓ {pdf_path.relative_to(ROOT)}')
print(f'\nFull bundle: {OUT_DIR.relative_to(ROOT)}')
print(f'  - action_list.csv')
print(f'  - bus_stops_report.pdf')
print(f'  - maps/*.html  (open any in a browser)')


---
## Wrap-up — what you now have

Starting from a single CSV of bus stops you now have:

| Artifact | Step | Audience |
|----------|------|----------|
| Temperature-joined stops table | 3 | GIS / analytics team |
| Visual hotspot ranking on the city map | 4 | Council presentation |
| Above-mean hotspot cluster map | 5 | Council presentation |
| Per-stop satellite diagnosis (top-N) | 6 | Landscape / infrastructure team |
| Per-stop street-level diagnosis (top-N) | 7 | Design review |
| Per-stop diurnal heat-index profile (top-N) | 8 | Intervention-type selection |
| Prioritized action list CSV | 9 | Operations hand-off |
| **Bundled hand-off folder (CSV + PDF report + interactive maps)** | **10** | **Slide decks, council packet, ops** |

Every action in the final list is traceable back to a measurement — not an assumption. That is the defensibility the council was missing, and the reason this workflow scales beyond bus stops.

### Where everything lives on disk

```
data/
  heatmaps/       ← raw heatmap GeoJSON outputs (live + cached)
  satellite/      ← raw satellite-segmentation JSON per stop
  street_view/    ← raw street-view JSON per stop
  env_params/     ← raw env-params JSON per stop
outputs/
  bus_stops_<STUDY_DATE>/
    action_list.csv
    bus_stops_report.pdf      ← multi-page PDF for slide decks
    maps/*.html               ← interactive folium maps
```

Re-running the notebook against any captured live response is a one-liner: change the filename in the matching cache cell (Step 2b / 6b / 7b / 8b) to point at any file in the corresponding `data/<type>/` subfolder.

### Apply this pattern to your other layers

The workflow is agnostic to what kind of asset your points represent. Swap the CSV for any point layer and everything downstream works:

- **Schools / playgrounds** → prioritize which outdoor spaces need tree planting
- **Public benches / transit shelters** → identify which need upgrading to reflective / vented designs
- **Bike-share docks** → identify stations where riders drop off because they overheat
- **Utility substations / pumping stations** → identify infrastructure at heat-failure risk
- **Social-housing units** → identify buildings in hottest blocks for retrofit prioritization

The pattern — **your geometries × our thermal, surface, and environmental layers → ranked actions** — is the whole point.